In [3]:
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

from nltk import tokenize

import math

In [ ]:
class MaksedMultiHeadAttention(nn.Module):
    def __init__(self,embed_dim, num_heads):
        super().__init__()
        if embed_dim%num_heads != 0 :
            raise ValueError(f"cant divide embed_dim{embed_dim} into {num_heads} heads")

        self.num_heads = num_heads
        self.q = nn.Linear(embed_dim,embed_dim)
        self.k = nn.Linear(embed_dim,embed_dim)
        self.v = nn.Linear(embed_dim,embed_dim)
        self.Wo = nn.Linear(embed_dim,embed_dim)


    def forward(self, input_batch): 
        B, T, E = input_batch.shape
        H = self.num_heads
        Eh = E//self.num_heads

        query_vec = self.q(input_batch)  # shape (B,T,E)x(ExE) = BxTxE
        key_vec = self.k(input_batch)    # shape (B,T,E)x(ExE) = BxTxE
        value_vec = self.v(input_batch)  # shape (B,T,E)x(ExE) = BxTxE

        head_q_vecs = query_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh) 
        head_k_vecs = key_vec.reshape(B,T,H,Eh).transpose(1,2)      #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)
        head_v_vecs = value_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)

        sim_scores = head_q_vecs @ head_k_vecs.transpose(-2,- 1) # (B,H,T,Eh) . (B,H,Eh,T) = (B,H,T,T)
        sim_scores = sim_scores/math.sqrt(Eh) #(B,H,T,T)

        attention = sim_scores @ head_v_vecs # (B,H,T,T).(B,H,T,Eh) = (B,H,T,Eh)

        out = attention.transpose(1,2).reshape(B,T,E) # from trnaspose : (B,T,H,Eh), from reshape : (B,T,E)
        output = self.Wo(out)
        return out
        